<!-- nav -->
[← 3 · The smallest expression, a literal](03_The_Smallest_Expression_A_Literal.ipynb) · [Home](../../README.md) · [5 · Operators →](05_Operators.ipynb)

# 4 · Names

After a literal, the next smallest expression is a name: `x`, a value the expression doesn't hold but finds in its
**scope** when it is simplified. This notebook models a name in each language, asks a term which names it needs, and
shows that a name is kept exactly as written, even one its language can't spell.

The code starts from `dialects()`, the toolkit's store with every dialect installed and defined. `S` is its
holders' namespace, `store.Expressions`, and `amd64` the ABI dialect C's and C++'s holders name, as in tutorial 3.

In [1]:
from mbse.Expressions import Translators
from mbse.Expressions.C import Expressions as C
from mbse.Expressions.Cpp import Expressions as Cpp
from mbse.Expressions.Excel import Expressions as Excel
from mbse.Expressions.Framework import Symbolics, Terms
from mbse.Expressions.LaTeX import Expressions as Latex
from mbse.Expressions.MATLAB import Expressions as Matlab
from mbse.Expressions.Patterns.Expressions import Simplifiers, Expressions as E
from mbse.Expressions.Python import Python3
from mbse.Expressions.SystemVerilog import Expressions as SystemVerilog
from mbse.Expressions.TypeScript import TypeScript5
from toolkit import dialects, show

store = dialects()
S, Term = store.Expressions, store.Expressions.Framework.Term
amd64 = store.Instances.Expressions.SystemV.AMD64

<a id="T04.1"></a>
## 04.1 · A variable

Patterns' `x` is a term of kind `variable`, whose one attribute is its `name`. It holds no value: simplification looks the
name up in the scope it's given, and replaces it by the literal of its value, here `{"x": 42}`:

In [2]:
with store.transaction():
    x = S.Patterns.Expressions.Expression(lambda e: e.root(lambda u: u.variable(lambda v: v.name("x"))))
form = Terms.form(Terms.term(x))
print(form.kind, form.attributes["name"], show(x))
print(Simplifiers.OfAny(x, {"x": 42}).value)

variable x x
42


Given no scope that binds `x`, simplification leaves it as it is: `x` stays in the residual, among the names it still
needs (`free`), a question for later rather than a mistake. Validation reports the same name, without simplifying:
`validate` takes the names the caller will bind, and reports every other name the term reads.

In [3]:
unbound = Simplifiers.OfAny(x, {})
print(sorted(unbound.free), unbound.literal)  # x stays: the residual is x itself, no literal
print(E.DIALECT.validate(x))
print(E.DIALECT.validate(x, bound=["x"]))

['x'] False
["variable 'x' is not bound"]
[]


<a id="T04.2"></a>
## 04.2 · A name in each language

Each language's dialect has its own kind for a name, each printed `x`:

- **Python3**'s `variable` (`ast`'s `Name`), and **C**'s, **Excel**'s and **MATLAB**'s: each the framework's `variable`,
  the kind Patterns' is too.
- **TypeScript5**'s `identifier`, as TypeScript's own syntax tree calls it.
- **C++**'s `idexpression` and **SystemVerilog**'s `nameexpression`, which have no `variable`
  ([CAP-230: the C++ expression dialect](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-230.md), [CAP-231: the SystemVerilog expression dialect, complete](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-231.md)): each holds a **name**, a helper of its own, since a name in those languages need not be a single
  identifier (`std::cout`, `f<int>`, `p::x`, `$unit::x`). Here it is the simplest, an `identifier`, its `spelling`
  `x`.
- **Excel**'s `variable` (a `LET` name or a defined name) beside its `cell`, `A1`, a reference no scope binds: the
  workbook resolves it.

Each model is built in its dialect's holder, and its dialect's `Text.ToText` gives the language's own text back.

Python's text is also read: `Python3.Text.FromText` reads `x` into the same term the builders wrote.

In [4]:
identifier = lambda spelling: lambda n: n.identifier(lambda i: i.spelling(spelling))  # noqa: E731: a C++ or SystemVerilog name
with store.transaction():
    names = [
        (Python3, S.Python.Python3.Expression(lambda e: e.root(lambda u: u.variable(lambda v: v.name("x"))))),
        (TypeScript5, S.TypeScript.TypeScript5.Expression(lambda e: e.root(lambda u: u.identifier(lambda i: i.name("x"))))),
        (C, S.C.Expressions.Expression(lambda e: e.root(lambda u: u.variable(lambda v: v.name("x"))).abi([amd64]))),
        (Cpp, S.Cpp.Expressions.Expression(lambda e: e.root(
            lambda u: u.idexpression(lambda r: r.name(identifier("x")))).abi([amd64]))),
        (SystemVerilog, S.SystemVerilog.Expressions.Expression(lambda e: e.root(
            lambda u: u.nameexpression(lambda r: r.name(identifier("x")))))),
        (Excel, S.Excel.Expressions.Expression(lambda e: e.root(lambda u: u.variable(lambda v: v.name("x"))))),
        (Excel, S.Excel.Expressions.Expression(lambda e: e.root(lambda u: u.cell(lambda c: c.address("A1"))))),
        (Matlab, S.MATLAB.Expressions.Expression(lambda e: e.root(lambda u: u.variable(lambda v: v.name("x"))))),
    ]
for dialect, name in names:
    print(f"{dialect.Expressions.DIALECT.name():13} {Terms.form(Terms.term(name)).kind:14} {dialect.Text.ToText(name)}")
assert [dialect.Text.ToText(name) for dialect, name in names] == ["x", "x", "x", "x", "x", "=x", "=A1", "x"]
with store.transaction():  # Python's text, read
    read = Term(Python3.Text.FromText("x"))
assert Terms.same(read, names[0][1])

Python3       variable       x
TypeScript5   identifier     x
C             variable       x
Cpp           idexpression   x
SystemVerilog nameexpression x
Excel         variable       =x
Excel         cell           =A1
MATLAB        variable       x


A C++ or SystemVerilog name is the term's `name` attribute, itself a part with a kind and attributes; `Terms.name_of`
reads the spelling the term refers by, whatever kind holds it:

In [5]:
for dialect, name in names[3:5]:
    form = Terms.form(Terms.term(name))
    part = form.attributes["name"]
    print(f"{dialect.Expressions.DIALECT.name():13} {form.kind} holds {part.kind} '{part.attributes['spelling']}'")
print(" ".join(Terms.name_of(Terms.term(name)) or "-" for _, name in names))

Cpp           idexpression holds identifier 'x'
SystemVerilog nameexpression holds identifier 'x'
x x x x x x - x


<a id="T04.3"></a>
## 04.3 · Free names

`Symbolics.free(term, DIALECT)` is what a term needs from its scope: the names it reads that nothing within it binds,
read as its dialect reads them. Python's builtins are always bound, so `len(x)` needs `x` alone, and validation reports
`x`, not `len`:

In [6]:
with store.transaction():
    length = S.Python.Python3.Expression(lambda e: e.root(lambda u: u.call(lambda c: c
        .func(lambda f: f.variable(lambda v: v.name("len"))).args([lambda a: a.variable(lambda v: v.name("x"))]))))
    read = Term(Python3.Text.FromText("len(x)"))  # Python's text, read
assert Python3.Text.ToText(length) == "len(x)" and Terms.same(read, length)
print(Python3.Text.ToText(length), "needs", ", ".join(sorted(Symbolics.free(length, Python3.Expressions.DIALECT))))
print(Python3.Expressions.DIALECT.validate(length))

len(x) needs x
["argument 0: variable 'x' is not bound"]


TypeScript's `undefined` is not a keyword but an `identifier`, as TypeScript itself parses it: a global the scope
provides, so a term that reads it needs it:

In [7]:
with store.transaction():
    undefined = S.TypeScript.TypeScript5.Expression(lambda e: e.root(lambda u: u.identifier(lambda i: i.name("undefined"))))
form = Terms.form(Terms.term(undefined))
print(form.kind, TypeScript5.Text.ToText(undefined), "needs",
      ", ".join(sorted(Symbolics.free(undefined, TypeScript5.Expressions.DIALECT))))

identifier undefined needs undefined


<a id="T04.4"></a>
## 04.4 · A name kept as written

`max-temp` is no name in Python: Python would read it as `max - temp`. Built as a name, it is kept in the tree as
written, printed as written, and reported by validation, never renamed ([PRI-002: preserve, don't lose](https://github.com/pitaman71/mbse-basis/blob/main/docs/PRINCIPLES.md#PRI-002)):

In [8]:
with store.transaction():
    max_temp = S.Python.Python3.Expression(lambda e: e.root(lambda u: u.variable(lambda v: v.name("max-temp"))))
print(Terms.form(Terms.term(max_temp)).attributes["name"], Python3.Text.ToText(max_temp))
print(Python3.Expressions.DIALECT.validate(max_temp, bound=["max-temp"]))

max-temp max-temp
["a name must be an identifier, got 'max-temp'"]


Each language that checks its names reports one it can't spell, in its own words; none renames it:

In [9]:
with store.transaction():
    unspellable = [
        (TypeScript5, S.TypeScript.TypeScript5.Expression(
            lambda e: e.root(lambda u: u.identifier(lambda i: i.name("max-temp"))))),
        (C, S.C.Expressions.Expression(lambda e: e.root(lambda u: u.variable(lambda v: v.name("max-temp"))).abi([amd64]))),
        (Cpp, S.Cpp.Expressions.Expression(lambda e: e.root(
            lambda u: u.idexpression(lambda r: r.name(identifier("max-temp")))).abi([amd64]))),
        (SystemVerilog, S.SystemVerilog.Expressions.Expression(lambda e: e.root(
            lambda u: u.nameexpression(lambda r: r.name(identifier("max-temp")))))),
    ]
for dialect, name in unspellable:
    print(f"{dialect.Expressions.DIALECT.name():13} {dialect.Text.ToText(name):9}",
          dialect.Expressions.DIALECT.validate(name, bound=["max-temp"]))

TypeScript5   max-temp  ["an identifier's name cannot be 'max-temp'"]
C             max-temp  ["'max-temp' is not an identifier"]
Cpp           max-temp  ["'max-temp' is not an identifier"]
SystemVerilog max-temp  ["'max-temp' is not an identifier"]


<a id="T04.5"></a>
## 04.5 · Names across languages

A translator ([CAP-101: translation between dialects](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-101.md)) writes `x` as each language's own name, the same term 04.2 built, and translates it back to
Patterns' `variable`. TypeScript5 has no translator yet, and says so:

In [10]:
built = {dialect.Expressions.DIALECT.name(): name for dialect, name in reversed(names)}  # Excel's variable, not its cell
for dialect in (Python3, TypeScript5, C, Cpp, SystemVerilog, Excel, Matlab, Latex):
    target = dialect.Expressions.DIALECT
    try:
        with store.transaction():
            there = Term(Translators.between(E.DIALECT, target).forward(x))
            back = Term(Translators.between(target, E.DIALECT).forward(there))
    except LookupError as error:
        print(f"{target.name():13} {type(error).__name__}: {error.args[0]}")
        continue
    same = Terms.same(there, built[target.name()]) if target.name() in built else "-"
    print(f"{target.name():13} {dialect.Text.ToText(there):3} {Terms.form(there).kind:14} {same!s:5} back: {show(back)}",
          Terms.same(back, x))

Python3       x   variable       True  back: x True
TypeScript5   LookupError: no translator between Patterns and TypeScript5
C             x   variable       True  back: x True
Cpp           x   idexpression   True  back: x True
SystemVerilog x   nameexpression True  back: x True
Excel         =x  variable       True  back: x True
MATLAB        x   variable       True  back: x True
LaTeX         x   variable       -     back: x True


## Why it works this way

- **A name is a reference, not a value.** A term that reads `x` holds the name, and the scope gives its value, so the
  same constraint is simplified against any data that binds its names, and `free` says what that data must bind.
- **Each language's own kind.** A dialect models its language's syntax tree as that language has it: TypeScript's
  `identifier`, C++'s `idexpression`, SystemVerilog's `nameexpression`, so that its text prints back as written.
- **Kept as written, reported, never repaired.** A name that a language can't spell is kept, printed and reported, so
  nothing in a model is lost or silently changed on its way through a tool.

<!-- nav -->
[← 3 · The smallest expression, a literal](03_The_Smallest_Expression_A_Literal.ipynb) · [Home](../../README.md) · [5 · Operators →](05_Operators.ipynb)